# Nepali TTS — Colab training shift (checkpoint relay) — **Stage B** (527 speakers / ~112h)

Lets a **free Colab GPU** take a training shift on the *same* run as the home laptop. The full-state
checkpoint (the "baton") is passed through a private Hugging Face repo; a lock stops both machines
from training at once; and the uploader refuses to overwrite a newer cloud checkpoint, so **no
progress is ever lost**. Stage B uses its **own** ckpt repo (`nepali-tts-ckpt-b`) so Stage-A
artifacts and epoch guards are untouched.

**Before running:** Runtime ▸ Change runtime type ▸ **GPU**; add your HF token as a Colab secret
(🔑 left sidebar) named `HF_TOKEN`; then Runtime ▸ **Run all**. Keep the tab open.

> Run the cells **in order**. Cell 6 refuses to train unless cell 5 actually took the lock.
> First-time setup from the laptop: `scripts/seed_stageB_hf.sh` (code+config+seed ckpt), then
> `scripts/upload_stageB_data.sh` (the ~13GB data tarball).

In [ ]:
# 1) Check we actually got a GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())

In [ ]:
# 2) Credentials + repo names (STAGE B: separate ckpt repo so Stage-A locks/epochs never clash)
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('token loaded from Colab secret')
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('Paste your HF write token: ')
os.environ['HF_REPO']   = 'byapaksigdel/nepali-tts-ckpt-b'
os.environ['DATA_REPO'] = 'byapaksigdel/nepali-tts-data'
os.environ['DEVICE_ID'] = 'colab'
os.environ['LOCK_STALE'] = '900'
print('repo:', os.environ['HF_REPO'])

In [ ]:
# 3) Fetch our code from the HF repo (token-auth, no GitHub clone) + espeak-ng + piper1-gpl trainer
import os, shutil
from huggingface_hub import hf_hub_download
os.makedirs('/content/nepali-tts/scripts', exist_ok=True)
os.makedirs('/content/nepali-tts/configs', exist_ok=True)
for src, dst in [('code/hf_sync.py', '/content/nepali-tts/scripts/hf_sync.py'),
                 ('code/status_writer.py', '/content/nepali-tts/scripts/status_writer.py'),
                 ('code/train_ne_stageB_colab.yaml', '/content/nepali-tts/configs/train_ne_stageB_colab.yaml')]:
    shutil.copy(hf_hub_download(os.environ['HF_REPO'], src, token=os.environ['HF_TOKEN']), dst)
print('code fetched from HF repo')
!apt-get -qq install -y espeak-ng >/dev/null && echo 'espeak-ng OK'
!pip -q install soundfile ninja
# Fresh clone PINNED to the commit our patch targets, so `git apply` is always clean
# (rm -rf makes this cell safe to re-run). Patch = atomic cache writes + self-healing reads
# (a truncated .pt must never kill a run) + smart warm-start + onnx dynamo fix.
!rm -rf /content/piper1-gpl
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!git checkout -q 2a60c2b
!curl -sL https://raw.githubusercontent.com/ByapakSigdel/nepali-tts/main/patches/piper_local.patch -o /content/piper_local.patch
!git apply /content/piper_local.patch && echo 'piper patches applied' || echo '!! PATCH FAILED — tell Claude'
!echo "atomic-save sites (expect 6): $(grep -c _atomic_torch_save src/piper/train/vits/dataset.py)"
!pip -q install -e '.[train]'
!bash ./build_monotonic_align.sh
!python setup.py build_ext --inplace >/dev/null 2>&1 && echo 'build_ext OK'
import torch; print('torch:', torch.__version__, '| gpu', torch.cuda.is_available())
!python -c 'import piper.train; print("piper.train import OK")'
%cd /content/nepali-tts

In [ ]:
# 4) Download the Stage-B training audio (once per session) from the private dataset repo
import os, tarfile
from huggingface_hub import hf_hub_download
p = hf_hub_download(os.environ['DATA_REPO'], 'processed_b.tar.gz', repo_type='dataset', token=os.environ['HF_TOKEN'])
os.makedirs('/content/data', exist_ok=True)
with tarfile.open(p) as t:
    t.extractall('/content/data')
print('dataset ready:', len(os.listdir('/content/data/processed_b/wavs')), 'clips')

In [ ]:
# 5) Take the lock + pull the baton (config + latest checkpoint). FAILS LOUDLY if it can't.
import os, sys, subprocess
SCRIPTS = '/content/nepali-tts/scripts'
RUN = '/content/run/ne_stageB'
CKPT = f'{RUN}/ckpts/last.ckpt'
os.makedirs(f'{RUN}/ckpts', exist_ok=True)
def hs(*a):
    return subprocess.run([sys.executable, f'{SCRIPTS}/hf_sync.py', *a], env=os.environ).returncode

CLAIMED = False
rc = hs('claim')
assert rc == 0, ('Could not take the training lock (another machine holds it, or state is unknown).\n'
                 'Stop the laptop relay first, or wait ~15 min for the lock to auto-free, then re-run.')
CLAIMED = True

assert hs('pull', 'config.json', f'{RUN}/config.json') == 0 and os.path.exists(f'{RUN}/config.json'), \
    'config.json pull failed'
assert hs('pull', 'last.ckpt', CKPT) == 0, 'checkpoint pull failed (network error) — re-run'
assert os.path.exists(CKPT) and os.path.getsize(CKPT) > 0, (
    'No checkpoint in the cloud yet. Seed it once from the laptop '
    '(scripts/seed_stageB_hf.sh) before training on Colab.')
print('baton acquired; remote epoch ->', end=' '); hs('remote-epoch')

In [ ]:
# 6) TRAIN (Stage B: 527 speakers / ~112h). An independent heartbeat thread keeps the lock alive
#    (never starved by the big push); the pusher uploads checkpoint+config ATOMICALLY and refuses
#    to overwrite newer cloud work. finally: pushes a final time and releases the lock ONLY if that
#    push landed.
#    NOTE: the first run of a session re-caches all 128k clips before training (~1.5-3h on Colab's
#    CPU — watch for 'Processing utterances...'); the patched trainer survives any corrupt cache.
import os, sys, re, time, threading, subprocess
SCRIPTS = '/content/nepali-tts/scripts'
RUN = '/content/run/ne_stageB'
CKPT = f'{RUN}/ckpts/last.ckpt'; CONFIG = f'{RUN}/config.json'; STATUS = f'{RUN}/status.txt'
os.environ['PYTHONPATH'] = SCRIPTS + ':' + os.environ.get('PYTHONPATH', '')
TARGET = 100
assert globals().get('CLAIMED'), 'Run cell 5 first — do NOT train without holding the lock.'

def hs(*a):
    return subprocess.run([sys.executable, f'{SCRIPTS}/hf_sync.py', *a], env=os.environ).returncode
def epoch_now():
    try:
        m = re.search(r'(?:^| )epoch=(\d+)', open(STATUS).read())
        return int(m.group(1)) if m else -1
    except Exception:
        return -1

_stop = False
def _sleep_interruptible(seconds):
    for _ in range(int(seconds / 2)):
        if _stop:
            return False
        time.sleep(2)
    return not _stop
def heartbeater():
    while _sleep_interruptible(120):
        hs('heartbeat')
def pusher():
    while _sleep_interruptible(1200):
        if os.path.exists(CKPT):
            hs('push-ckpt', CKPT, 'auto', '--config', CONFIG)
threading.Thread(target=heartbeater, daemon=True).start()
threading.Thread(target=pusher, daemon=True).start()

cmd = ['python', '-m', 'piper.train', 'fit',
       '--config', '/content/nepali-tts/configs/train_ne_stageB_colab.yaml',
       '--data.voice_name', 'ne_stageB',
       '--data.csv_path', '/content/data/processed_b/metadata.csv',
       '--data.audio_dir', '/content/data/processed_b/wavs',
       '--data.espeak_voice', 'ne',
       '--data.cache_dir', f'{RUN}/cache',
       '--data.config_path', f'{RUN}/config.json',
       '--data.batch_size', '8',
       '--data.num_workers', '2',
       '--model.sample_rate', '22050',
       '--model.num_speakers', '527',
       '--trainer.accelerator', 'gpu', '--trainer.devices', '1',
       '--trainer.precision', '32-true',
       '--trainer.max_epochs', str(TARGET),
       '--trainer.default_root_dir', RUN,
       '--trainer.num_sanity_val_steps', '0',
       '--trainer.limit_val_batches', '0',
       '--trainer.log_every_n_steps', '25',
       '--ckpt_path', CKPT]

print('Training Stage B on Colab GPU -> epoch', TARGET, '. Keep this tab open.')
try:
    for attempt in range(1, 100):
        if epoch_now() >= TARGET:
            print('reached target'); break
        print(f'--- attempt {attempt} ---')
        code = subprocess.run(cmd, env=os.environ).returncode
        if code == 0:
            print('training finished cleanly'); break
        print(f'exited {code}; resuming in 8s'); time.sleep(8)
finally:
    _stop = True
    final_ok = True
    if os.path.exists(CKPT):
        final_ok = False
        for a in range(5):
            if hs('push-ckpt', CKPT, 'auto', '--config', CONFIG) == 0:
                final_ok = True; break
            time.sleep(15)
    if final_ok:
        hs('release'); print('FINAL push done + lock released. Safe to close. epoch =', epoch_now())
    else:
        print('!!! FINAL PUSH FAILED — KEEPING the lock so the laptop cannot overwrite your work.')
        print('    Your epochs are safe in this VM at', CKPT, '— re-run THIS cell when the network is back.')

## When you're done (or Colab kicks you off)

The last cell pushed the checkpoint and released the lock, so the laptop can pull the progress and
continue. If Colab disconnected abruptly, at most the last ~20 minutes (since the previous auto-push)
is re-done; nothing is lost, and the uploader can never overwrite newer work.

Check the lock from anywhere: `!python /content/nepali-tts/scripts/hf_sync.py status`